In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')
import config
from src.data_loader import load_data
from src.pair_selector import (
    run_pair_selection, get_selected_pairs,
    compute_spread, test_cointegration, compute_half_life,
    test_cointegration_stability, generate_candidate_pairs
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Load cached price data (no re-download needed)
prices = load_data(force_refresh=False)
print(f"Loaded {prices.shape[1]} tickers, {prices.shape[0]} trading days")

## 1. Run the Pair Selection Pipeline

This scans all ~300 intra-sector pairs and for each one:
- Runs OLS regression to get the hedge ratio (β) and spread
- Tests the spread for stationarity (ADF test)
- Computes mean-reversion half-life
- Checks cointegration stability across rolling windows
- Scores and ranks everything

**Important:** This uses only the TRAINING data (first ~3 years).
The last year is held out for backtesting — never touched here.

In [ ]:
# Run full pipeline — takes 1-3 minutes depending on your machine
results = run_pair_selection(prices, verbose=True)

In [ ]:
# Look at all results
print(f"\nTotal pairs analyzed: {len(results)}")
print(f"Cointegrated pairs (ADF p < 0.05): {results['is_cointegrated'].sum()}")
print(f"Non-cointegrated: {(~results['is_cointegrated']).sum()}")
print(f"\nBreakdown by sector:")
print(results.groupby('sector')['is_cointegrated'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'Total Pairs', 'sum': 'Cointegrated', 'mean': 'Coint Rate'}
).round(2))

In [ ]:
# Show top 20 pairs by score
top_cols = ['stock_a', 'stock_b', 'sector', 'beta', 'adf_pvalue', 
            'half_life', 'stability_pct', 'score']
results.head(20)[top_cols]

## 2. Visualize: ADF P-values vs Half-Life

This scatter plot shows all pairs in two dimensions:
- **X-axis:** ADF p-value (lower = more stationary = better)
- **Y-axis:** Half-life in days (5-30 is the sweet spot)

The best pairs are in the **bottom-left corner** (low p-value, reasonable half-life).
The green shaded region shows our selection criteria.

In [ ]:
# Filter for plottable pairs (positive half-life only)
plot_df = results[results['half_life'] > 0].copy()
plot_df['pair_name'] = plot_df['stock_a'].str.replace('.NS','') + ' / ' + plot_df['stock_b'].str.replace('.NS','')

fig = px.scatter(
    plot_df, 
    x='adf_pvalue', y='half_life',
    color='sector', 
    hover_name='pair_name',
    hover_data=['beta', 'stability_pct', 'score'],
    title='Pair Selection: ADF P-value vs Half-Life',
    labels={'adf_pvalue': 'ADF P-value (lower = better)', 'half_life': 'Half-Life (days)'},
    template='plotly_dark',
    height=600, width=900,
)

# Add selection region (green box)
fig.add_vrect(x0=0, x1=config.ADF_P_VALUE_CUTOFF, 
              fillcolor='green', opacity=0.08, line_width=0)
fig.add_hrect(y0=config.HALF_LIFE_MIN, y1=config.HALF_LIFE_MAX, 
              fillcolor='green', opacity=0.08, line_width=0)

# Add threshold lines
fig.add_vline(x=config.ADF_P_VALUE_CUTOFF, line_dash='dash', line_color='red', 
              annotation_text=f'p = {config.ADF_P_VALUE_CUTOFF}')
fig.add_hline(y=config.HALF_LIFE_MIN, line_dash='dash', line_color='orange')
fig.add_hline(y=config.HALF_LIFE_MAX, line_dash='dash', line_color='orange',
              annotation_text=f'HL = {config.HALF_LIFE_MAX}d')

fig.show()

In [ ]:
selected = get_selected_pairs(results)
selected[top_cols]

## 4. Deep Dive: Visualize Each Selected Pair

For each selected pair, we'll show:
- **Top:** Both stock prices (normalized to 100)
- **Bottom:** The spread with its mean — you should see it oscillating

In [ ]:
# Use training data only (same as pair selection used)
train_prices = prices.iloc[:-config.TEST_PERIOD_DAYS]

for _, row in selected.iterrows():
    a, b = row['stock_a'], row['stock_b']
    a_name = a.replace('.NS', '')
    b_name = b.replace('.NS', '')
    
    price_a = train_prices[a]
    price_b = train_prices[b]
    
    # Compute spread
    ols = compute_spread(price_a, price_b)
    spread = ols['spread']
    
    # Normalize prices for comparison
    norm_a = price_a / price_a.iloc[0] * 100
    norm_b = price_b / price_b.iloc[0] * 100
    
    # Create subplot
    fig = make_subplots(rows=2, cols=1, row_heights=[0.5, 0.5],
                        subplot_titles=[f'{a_name} vs {b_name} (Normalized)', 'Spread (OLS Residuals)'])
    
    # Price chart
    fig.add_trace(go.Scatter(x=norm_a.index, y=norm_a, name=a_name, line=dict(color='#00d4aa')), row=1, col=1)
    fig.add_trace(go.Scatter(x=norm_b.index, y=norm_b, name=b_name, line=dict(color='#ff6b6b')), row=1, col=1)
    
    # Spread chart
    fig.add_trace(go.Scatter(x=spread.index, y=spread, name='Spread', line=dict(color='#6b9fff')), row=2, col=1)
    fig.add_hline(y=spread.mean(), row=2, col=1, line_dash='dash', line_color='yellow',
                  annotation_text='Mean')
    fig.add_hline(y=spread.mean() + 2*spread.std(), row=2, col=1, line_dash='dot', line_color='red')
    fig.add_hline(y=spread.mean() - 2*spread.std(), row=2, col=1, line_dash='dot', line_color='red')
    
    fig.update_layout(
        height=500, template='plotly_dark',
        title=f"{a_name} ↔ {b_name} ({row['sector']}) | β={row['beta']:.3f} | ADF p={row['adf_pvalue']:.4f} | HL={row['half_life']:.0f}d"
    )
    fig.show()

## 5. Cointegration Stability Over Time

For the top pairs, let's see if cointegration holds consistently.
We run ADF tests on rolling 1-year windows and plot the p-values.

Stable pairs will have p-values consistently below 0.05.
Unstable pairs will have p-values that spike above 0.05 frequently.

In [ ]:
# Stability analysis for top pairs
n_pairs_to_show = min(6, len(selected))

fig = make_subplots(rows=n_pairs_to_show, cols=1,
                    subplot_titles=[
                        f"{row['stock_a'].replace('.NS','')} / {row['stock_b'].replace('.NS','')}"
                        for _, row in selected.head(n_pairs_to_show).iterrows()
                    ])

for i, (_, row) in enumerate(selected.head(n_pairs_to_show).iterrows(), 1):
    stab = test_cointegration_stability(
        train_prices[row['stock_a']], 
        train_prices[row['stock_b']]
    )
    
    fig.add_trace(go.Scatter(
        y=stab['pvalues'], 
        name=f"Window p-values",
        mode='lines+markers',
        line=dict(color='#6b9fff'),
        showlegend=False
    ), row=i, col=1)
    
    # Add p=0.05 threshold
    fig.add_hline(y=0.05, row=i, col=1, line_dash='dash', line_color='red')
    
    fig.update_yaxes(title_text='ADF p-value', row=i, col=1)

fig.update_layout(
    height=250 * n_pairs_to_show, 
    template='plotly_dark',
    title='Cointegration Stability: Rolling ADF P-values (below red line = cointegrated)'
)
fig.show()

## 6. Understanding the Rejected Pairs

Let's also look at WHY most pairs got rejected. This builds intuition
for what makes a good pair.

In [ ]:
# Distribution of ADF p-values
fig = px.histogram(
    results, x='adf_pvalue', nbins=30,
    title='Distribution of ADF P-values Across All Pairs',
    labels={'adf_pvalue': 'ADF P-value'},
    template='plotly_dark',
    color_discrete_sequence=['#6b9fff']
)
fig.add_vline(x=0.05, line_dash='dash', line_color='red', 
              annotation_text='p = 0.05 cutoff')
fig.update_layout(height=400)
fig.show()

pct_passed = results['is_cointegrated'].mean() * 100
print(f"\n📊 Only {pct_passed:.1f}% of pairs passed the ADF test.")

In [ ]:
# Compare a good pair vs a bad pair
if len(selected) > 0:
    # Best pair
    best = selected.iloc[0]
    # Worst pair (highest ADF p-value)
    worst = results.iloc[-1]
    
    fig = make_subplots(rows=2, cols=1, subplot_titles=[
        f"GOOD: {best['stock_a'].replace('.NS','')} / {best['stock_b'].replace('.NS','')} (ADF p={best['adf_pvalue']:.4f})",
        f"BAD: {worst['stock_a'].replace('.NS','')} / {worst['stock_b'].replace('.NS','')} (ADF p={worst['adf_pvalue']:.4f})",
    ])
    
    # Good pair spread
    good_spread = compute_spread(train_prices[best['stock_a']], train_prices[best['stock_b']])['spread']
    fig.add_trace(go.Scatter(x=good_spread.index, y=good_spread, name='Good Spread', 
                             line=dict(color='#00d4aa')), row=1, col=1)
    fig.add_hline(y=good_spread.mean(), row=1, col=1, line_dash='dash', line_color='yellow')
    
    # Bad pair spread
    bad_spread = compute_spread(train_prices[worst['stock_a']], train_prices[worst['stock_b']])['spread']
    fig.add_trace(go.Scatter(x=bad_spread.index, y=bad_spread, name='Bad Spread', 
                             line=dict(color='#ff6b6b')), row=2, col=1)
    fig.add_hline(y=bad_spread.mean(), row=2, col=1, line_dash='dash', line_color='yellow')
    
    fig.update_layout(height=500, template='plotly_dark',
                      title='Good vs Bad Pair: Spread Behavior')
    fig.show()

In [ ]:
# Save all results
results.to_csv('./data/pair_selection_results.csv', index=False)
selected.to_csv('./data/selected_pairs.csv', index=False)

print(f"✅ Saved {len(results)} pair results to data/pair_selection_results.csv")
print(f"✅ Saved {len(selected)} selected pairs to data/selected_pairs.csv")
print(f"\nSelected pairs for trading:")
for _, row in selected.iterrows():
    a = row['stock_a'].replace('.NS', '')
    b = row['stock_b'].replace('.NS', '')
    print(f"  {a} ↔ {b} ({row['sector']}) — Score: {row['score']:.3f}")